# Gaussian covariance barycenter grids

This notebook generates `fig:barycenters-gaussian-covariances`.  For centered Gaussian inputs, quadratic Wasserstein barycenters remain Gaussian.  The covariance matrix is the Bures--Wasserstein barycenter
$$
    \Sigma_{u,v}\in\arg\min_{\Sigma\succ0}\sum_{i,j}\lambda_{ij}(u,v)\,B(\Sigma,\Sigma_{ij})^2.
$$
The two exported panels show only the resulting $5\times5$ grids: a moderate configuration and a more anisotropic one.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")
for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Ellipse
from PIL import Image
import ot
from scipy.linalg import expm, solve
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter
from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY, DIRAC_MARKER_SIZE,
    setup_matplotlib, figure_dir, save_pdf, remove_axes, box_axes,
    interp_color, draw_point_clouds, draw_transport_segments, padded_limits,
)
setup_matplotlib()
rng = np.random.default_rng(2027)

In [2]:
NAME="barycenters-gaussian-covariances"; out=figure_dir(NAME)
def cov(angle, axes):
    c,s=np.cos(angle),np.sin(angle); R=np.array([[c,-s],[s,c]])
    return R@np.diag(np.asarray(axes)**2)@R.T
def weights(u,v): return np.array([(1-u)*(1-v), u*(1-v), (1-u)*v, u*v])
corner_colors=np.array([[0.839,0.188,0.153],[0.992,0.682,0.380],[0.482,0.196,0.580],[0.129,0.400,0.675]])
def color(u,v): return tuple(np.clip(weights(u,v)@corner_colors,0,1))
def ellipse(ax, mean, Sigma, color, scale=0.72):
    vals, vecs=np.linalg.eigh(Sigma); order=vals.argsort()[::-1]; vals=vals[order]; vecs=vecs[:,order]
    angle=np.degrees(np.arctan2(vecs[1,0],vecs[0,0]))
    e=Ellipse(mean, width=scale*np.sqrt(vals[0]), height=scale*np.sqrt(vals[1]), angle=angle, facecolor=color, edgecolor=color, lw=0.7, alpha=0.26)
    ax.add_patch(e)
    ax.add_patch(Ellipse(mean, width=scale*np.sqrt(vals[0]), height=scale*np.sqrt(vals[1]), angle=angle, facecolor='none', edgecolor=color, lw=0.85, alpha=0.95))
def draw_grid(covs, filename):
    uv=np.linspace(0,1,5)
    bary={}
    for v in uv:
        for u in uv:
            bary[(u,v)] = ot.gaussian.bures_barycenter_fixpoint(covs, weights=weights(float(u),float(v)), num_iter=120, eps=1e-12)
    maxstd=max(np.sqrt(np.linalg.eigvalsh(S).max()) for S in bary.values())
    fig,ax=plt.subplots(figsize=(2.9,2.9))
    for row,v in enumerate(uv[::-1]):
        for col,u in enumerate(uv):
            S=bary[(float(u),float(v))]
            ellipse(ax, np.array([col+0.5,row+0.5]), S/(maxstd**2), color(float(u),float(v)))
    ax.set_xlim(0,5); ax.set_ylim(0,5); ax.set_aspect('equal'); remove_axes(ax)
    save_pdf(fig,out/filename,pad_inches=0.04); plt.close(fig)
moderate=np.array([cov(-0.65,(0.90,0.28)), cov(0.36,(0.40,0.96)), cov(1.10,(0.78,0.25)), cov(-0.12,(0.58,0.48))])
anisotropic=np.array([cov(-0.80,(1.20,0.13)), cov(0.72,(0.18,1.18)), cov(1.32,(1.05,0.16)), cov(-0.25,(0.95,0.20))])
draw_grid(moderate,'moderate-grid.pdf')
draw_grid(anisotropic,'anisotropic-grid.pdf')

Dit not converge.
Dit not converge.
Dit not converge.
